In [61]:
from langsmith import traceable, Client
from tools import rag_tool
from chat_app_backend_rag import _load_and_split, generate_output, _get_vectorstore
from dotenv import load_dotenv
import os
from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langsmith import wrappers

In [32]:
get_path = Path(r"D:\AI-ML\AI Projects\fastapi-chat-app\eval_files\mml-book.pdf")

In [ ]:
load_dotenv() 
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.2, max_tokens=256)

In [ ]:
client = Client()
dataset = client.create_dataset(
    dataset_name="math-for-ml-eval_V2",
    description="Evaluation set covering core ML mathematics concepts"
)

In [ ]:
examples = [
    {
        "inputs": {"question": "What is an eigenvector, and what does its eigenvalue represent?"},
        "outputs": {"answer": "An eigenvector of a matrix A is a nonzero vector v such that Av = λv for some scalar λ. The eigenvalue λ represents how much the eigenvector is stretched or shrunk by the transformation, without changing its direction."}
    },
    {
        "inputs": {"question": "Why is the covariance matrix always symmetric and positive semi-definite?"},
        "outputs": {"answer": "It's symmetric because Cov(X,Y) = Cov(Y,X) by definition. It's positive semi-definite because for any vector v, v^T C v equals the variance of the linear combination v^T X, and variance can never be negative."}
    },
    {
        "inputs": {"question": "What does PCA maximize, and what does it minimize, when reducing dimensionality?"},
        "outputs": {"answer": "PCA maximizes the variance of the projected data along each principal component, which is mathematically equivalent to minimizing the reconstruction error (squared distance) between the original data points and their projections onto the lower-dimensional subspace."}
    },
    {
        "inputs": {"question": "What is the gradient of a scalar-valued function, and what direction does it point in?"},
        "outputs": {"answer": "The gradient is the vector of partial derivatives of the function with respect to each input variable. It points in the direction of steepest ascent of the function at that point, and its magnitude indicates the rate of increase in that direction."}
    },
    {
        "inputs": {"question": "What is the difference between a maximum likelihood estimate and a maximum a posteriori estimate?"},
        "outputs": {"answer": "MLE finds parameters that maximize the likelihood of the observed data alone, P(D|θ). MAP additionally incorporates a prior belief about the parameters, maximizing the posterior P(θ|D), which is proportional to P(D|θ)P(θ). MAP reduces to MLE when the prior is uniform."}
    },
    {
        "inputs": {"question": "What does it mean for a matrix to be positive definite, and why does this matter for optimization?"},
        "outputs": {"answer": "A matrix A is positive definite if v^T A v > 0 for all nonzero vectors v. In optimization, if the Hessian of a function is positive definite at a critical point, that point is a strict local minimum — this is how second-order conditions confirm minima versus maxima or saddle points."}
    },
    {
        "inputs": {"question": "What is the chain rule's role in backpropagation?"},
        "outputs": {"answer": "Backpropagation computes gradients of a loss function with respect to every parameter in a composed, multi-layer function. The chain rule lets this be done efficiently by multiplying local derivatives layer by layer backward through the network, rather than differentiating the entire composition directly."}
    },
    {
        "inputs": {"question": "What is the difference between L1 and L2 regularization in terms of the geometry of their constraint regions?"},
        "outputs": {"answer": "L2 regularization constrains weights to lie within a sphere (in the weight space), which tends to shrink weights smoothly toward zero. L1 regularization constrains weights to a diamond-shaped region with sharp corners aligned with the axes, which makes solutions more likely to land exactly on an axis — producing sparse solutions with some weights exactly zero."}
    },
]

inputs=[ex["inputs"] for ex in examples],
outputs=[ex["outputs"] for ex in examples],


In [16]:

client.create_examples(
    inputs=[ex["inputs"] for ex in examples],
    outputs=[ex["outputs"] for ex in examples],
    dataset_id=dataset.id,
)

{'example_ids': ['b3264707-9058-440c-bbfa-5de83b95b967',
  'fcc8a864-41c0-45e6-8d05-ef93542b42d8',
  '6f287a2f-f8e3-4c95-8aa2-cb6d579710e4',
  'd184c553-dfab-44ed-8851-5a5716d442a8',
  '4fae14f6-c911-4a78-bdc1-b7c7aa2ed6e2',
  'b6bfbf42-4985-453c-ac21-1cb7624563c0',
  '3c18296b-1236-435a-a69d-abc2e5f7e467',
  '169af907-11aa-4ed0-9d39-0f93b9560a03'],
 'count': 8,
 'as_of': '2026-09-16T16:02:58.526051481Z'}

In [51]:
from chat_app_backend_rag import _get_vectorstore, generate_output

def rag_tool(query: str) -> str:

    """Retrieve relevant passages from the user's uploaded documents to answer
    questions about their specific content — facts, data, names, dates,
    numbers, or details that live in those files rather than in general
    knowledge. This is the only way to access document content; you cannot
    see uploaded files directly.

    WHEN TO CALL:
    Call this tool exactly once when the user's question could plausibly be
    answered by their uploaded documents — including questions about people,
    projects, figures, or specifics you would otherwise have to guess at.

    WHEN NOT TO CALL:
    - Greetings, small talk, or conversational filler
    - Math, logic, or calculations (use the calculator tool instead)
    - General knowledge you already know with confidence
    - Follow-up questions about tone, formatting, or phrasing rather than facts
    - Any question already answered earlier in this conversation

    CALL LIMIT — READ CAREFULLY:
    Call this tool at most once per user question. If the result indicates no
    relevant documents were found, that is a final result, not a signal to
    retry. Do not call this tool again with a rephrased, broadened, or
    alternate query in the same turn. Instead, immediately answer using your
    own general knowledge and clearly tell the user their documents did not
    contain the relevant information. Repeated calls for the same question
    waste time and essentially never surface something a well-formed first
    query missed.

    Args:
        query: A precise, standalone search query capturing exactly what the
            user wants to find — not the user's raw message. Resolve pronouns
            and vague references into concrete terms (e.g. "what about its
            pricing?" becomes "product pricing details"). Keep it focused on
            one specific piece of information; do not bundle multiple
            unrelated questions into one query.

    Returns:
        The most relevant retrieved passages with source attribution, or an
        explicit message stating no relevant documents were found — treat the
        latter as final, not as a prompt to try again.
    """
    vector_store = _get_vectorstore()
    return generate_output(query, vector_store)

In [52]:
from chat_app_backend_rag import add_documents_to_store

get_path = Path(r"D:\AI-ML\AI Projects\fastapi-chat-app\eval_files\mml-book.pdf")
add_documents_to_store([str(get_path)], collection_name="file_embeddings")


add_documents_to_store([str(get_path)], collection_name="mml_eval")
# and in your local rag_tool:
vector_store = _get_vectorstore("mml_eval")

In [59]:
def check_answer(question: str, docs):
    check_prompt = PromptTemplate(
        template="""
            You are a helpfull assistent, give answer from following Query: {question},
            here is your refferance document: {docs}.
        """,
        input_variables=['question', 'docs']
    )
    prompt = check_prompt.format(question=question, docs=docs)
    return {"responce": llm.invoke(prompt).content}

In [60]:
gen_outputs = []
for ques in examples:
    # print(ques)

    referance = rag_tool(ques['inputs']['question'])
    gen_outputs.append(check_answer(ques, referance))

check_chunk_quality output: relevant_indeces=[0] relevant=True
[grading] failed, treating as no-match: Error code: 400 - {'error': {'message': 'Tool choice is required, but model did not call a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': ''}}
[grading] failed, treating as no-match: Error code: 400 - {'error': {'message': 'Tool choice is required, but model did not call a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': ''}}
check_chunk_quality output: relevant_indeces=[1, 2] relevant=True
check_chunk_quality output: relevant_indeces=[0, 1, 4] relevant=True
[grading] failed, treating as no-match: Error code: 400 - {'error': {'message': 'Tool choice is required, but model did not call a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': ''}}
[grading] failed, treating as no-match: Error code: 400 - {'error': {'message': 'Tool choice is required, but model did not call a

In [64]:
for ques in examples:
    print(ques['inputs']['question'])

What is an eigenvector, and what does its eigenvalue represent?
Why is the covariance matrix always symmetric and positive semi-definite?
What does PCA maximize, and what does it minimize, when reducing dimensionality?
What is the gradient of a scalar-valued function, and what direction does it point in?
What is the difference between a maximum likelihood estimate and a maximum a posteriori estimate?
What does it mean for a matrix to be positive definite, and why does this matter for optimization?
What is the chain rule's role in backpropagation?
What is the difference between L1 and L2 regularization in terms of the geometry of their constraint regions?


In [62]:
gen_outputs

[{'responce': '**Answer**\n\nAn eigenvector of a square matrix \\(A\\) is a non‑zero vector \\(v\\) that satisfies  \n\n\\[\nA v = \\lambda v\n\\]\n\nfor some scalar \\(\\lambda\\).  \nThe scalar \\(\\lambda\\) is called the **eigenvalue** corresponding to that eigenvector.  \n\nThe eigenvalue tells us how the linear transformation represented by \\(A\\) scales the eigenvector: it is the factor by which the eigenvector is stretched (if \\(|\\lambda|>1\\)), shrunk (if \\(|\\lambda|<1\\)), or left unchanged (if \\(\\lambda=1\\)). Importantly, the direction of the eigenvector does **not** change under the transformation; only its magnitude is altered by the factor \\(\\lambda\\).  \n\n*(Source: Section\u202f4.2 “Eigenvalues and Eigenvectors” in mml‑book.pdf.)*'},
 {'responce': '**Why the covariance matrix is always symmetric and positive‑semi‑definite**\n\nLet \\(X=(X_1,\\dots ,X_p)^{\\top}\\) be a random vector with finite second moments and mean vector \\(\\mu=E[X]\\).  \nIts covariance

In [37]:
eval_instructions = "You are am export professor specialized in grading students"

def correcrness(input:dict, output: dict, refferance_outputs: dict) -> bool:
    user_content = f"""
                You are grading the following question:
                {input['question']}
                Here is the real answer:
                {refferance_outputs['answer']}
                You are grading the following predicted answer:
                {output['responce']}
                Respond with CORRECT or INCORRECT:
                Grade:
            """
    response = llm.invoke(
                    input = [{"role":"system", "content": eval_instructions},
                            {"role": "user", "content": user_content}]
                    ).content

    return response == "CORRECT"

In [ ]:
def concision(output: dict, reference_output: dict) -> bool:
    return int(len(output["response"]) < 2 * len(reference_output['answer']))

In [69]:
default_instructions = "Respond to the users question is sort, concise manner (2-3 short sentance)."

def my_app(question: str, instruction: str = default_instructions) -> str:
    return llm.invoke(input = [{"role":"system", "content": instruction},
                                {"role": "user", "content": question}]
                    ).content

In [70]:
def ls_target(input: str) -> dict:
    return {"response": my_app(input["question"])}

In [74]:
generated_by_question = {
    example["inputs"]["question"]: generated["responce"]
    for example, generated in zip(examples, gen_outputs)
}

def stored_target(inputs: dict) -> dict:
    question = inputs["question"]
    return {"response": generated_by_question[question]}

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""
Question: {inputs["question"]}
Reference answer: {reference_outputs["answer"]}
Generated answer: {outputs["response"]}

Reply with only CORRECT or INCORRECT.
"""
    result = llm.invoke(user_content).content.strip().upper()
    return result.startswith("CORRECT")

def concision(outputs: dict, reference_outputs: dict) -> bool:
    response_words = len(outputs["response"].split())
    reference_words = len(reference_outputs["answer"].split())
    allowed_words = max(2 * reference_words, 80)
    return response_words <= allowed_words

experiment_result = client.evaluate(
    stored_target,
    data=dataset.name,
    evaluators=[correctness, concision],
    experiment_prefix="stored-gen-outputs",
)

experiment_result

View the evaluation results for experiment: 'stored-gen-outputs-139492d3' at:
https://smith.langchain.com/o/8b96c356-c6bf-4cbd-9d1b-e231816f9207/datasets/043ef9d6-f84c-4ae9-9a7f-5e3f0215bd55/compare?selectedSessions=ecb84068-9f35-42d1-9c87-14449b5f94eb




8it [00:05,  1.55it/s]


,inputs.question,outputs.response,error,reference.answer,feedback.correctness,feedback.concision,execution_time,example_id,id
0,What is the difference between L1 and L2 regul...,L2 regularization constrains the weight vector...,None,L2 regularization constrains weights to lie wi...,True,True,0.004934,169af907-11aa-4ed0-9d39-0f93b9560a03,01a0ab31-7365-7380-80bf-3e1e1951f26b
1,What is the chain rule's role in backpropagation?,Backpropagation computes gradients of a loss f...,None,Backpropagation computes gradients of a loss f...,True,True,0.000223,3c18296b-1236-435a-a69d-abc2e5f7e467,01a0ab31-75e1-7850-9fb5-0c425a8f79b0
2,What is the difference between a maximum likel...,**Maximum Likelihood Estimate (MLE)** \n- Loo...,None,MLE finds parameters that maximize the likelih...,True,False,0.000150,4fae14f6-c911-4a78-bdc1-b7c7aa2ed6e2,01a0ab31-780f-71c3-b7b3-4bc5add74a4f
3,"What does PCA maximize, and what does it minim...",PCA **maximizes** the variance captured by eac...,None,PCA maximizes the variance of the projected da...,True,True,0.000193,6f287a2f-f8e3-4c95-8aa2-cb6d579710e4,01a0ab31-7a74-7433-8892-0e691317aaea
4,"What is an eigenvector, and what does its eige...",**Answer**\n\nAn eigenvector of a square matri...,None,An eigenvector of a matrix A is a nonzero vect...,True,False,0.000201,b3264707-9058-440c-bbfa-5de83b95b967,01a0ab31-7d04-7de0-a3a8-e82620052181
5,What does it mean for a matrix to be positive ...,**Positive‑definite matrix**\n\nA real symmetr...,None,A matrix A is positive definite if v^T A v > 0...,True,False,0.000135,b6bfbf42-4985-453c-ac21-1cb7624563c0,01a0ab31-7f0c-79d1-92de-21ff869b297d
6,What is the gradient of a scalar-valued functi...,**Answer**\n\nThe gradient of a scalar‑valued ...,None,The gradient is the vector of partial derivati...,True,True,0.000129,d184c553-dfab-44ed-8851-5a5716d442a8,01a0ab31-81b2-7df1-9d0d-5307f23c0780
7,Why is the covariance matrix always symmetric ...,**Why the covariance matrix is always symmetri...,None,"It's symmetric because Cov(X,Y) = Cov(Y,X) by ...",True,False,0.000126,fcc8a864-41c0-45e6-8d05-ef93542b42d8,01a0ab31-8408-72c2-b0a5-e6fb1eb246f1


In [72]:
length_diagnostics = []
for example, generated in zip(examples, gen_outputs):
    reference = example["outputs"]["answer"]
    response = generated["responce"]
    length_diagnostics.append({
        "question": example["inputs"]["question"],
        "reference_chars": len(reference),
        "response_chars": len(response),
        "allowed_chars": 2 * len(reference),
        "ratio": round(len(response) / len(reference), 2),
        "passes": len(response) < 2 * len(reference),
    })

import pandas as pd
pd.DataFrame(length_diagnostics)

,question,reference_chars,response_chars,allowed_chars,ratio,passes
0,"What is an eigenvector, and what does its eige...",218,707,436,3.24,False
1,Why is the covariance matrix always symmetric ...,208,2050,416,9.86,False
2,"What does PCA maximize, and what does it minim...",265,332,530,1.25,True
3,What is the gradient of a scalar-valued functi...,247,613,494,2.48,False
4,What is the difference between a maximum likel...,268,1402,536,5.23,False
5,What does it mean for a matrix to be positive ...,283,2834,566,10.01,False
6,What is the chain rule's role in backpropagation?,306,306,612,1.00,True
7,What is the difference between L1 and L2 regul...,362,429,724,1.19,True
